# Text + metadata pipeline (Phase 3, issue I-3)

The paper claims a combined TF-IDF + one-hot-metadata feature space (speaker, subject,
party, context, job, state), but `proposed_improvements.ipynb` never builds it -- a
paper/code mismatch. This notebook implements Option A from `claude-workspace/ISSUE_PLAN.md`
Phase 3: it actually builds the metadata feature space (`metadata_features.py`) and
combines it with the TF-IDF text features, so the claim becomes true.

Three feature sets are compared, all on the label-corrected data and the same metric set
(macro-F1 primary) from Phase 1:
1. **Text only** -- carried over from `proposed_improvements_v2.ipynb` (Chi2 + MI).
2. **Text + metadata (no speaker)** -- subject/party/state/job/context, one-hot/multi-label.
3. **Text + metadata + speaker (hashed)** -- adds speaker via a 64-dim `FeatureHasher`
   rather than raw one-hot, to report its effect explicitly without giving the model a
   speaker-identity lookup table (see `metadata_features.py` docstring for why).

The five `*_counts` credit-history columns are never used (label leakage).

In [1]:
import re

import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

from liar_utils import RANDOM_STATE, evaluate_full, load_and_label, print_report
from metadata_features import build_metadata_features, combine_text_and_metadata

Load data (corrected labels) and preprocess text exactly as in `proposed_improvements_v2.ipynb`

In [2]:
train = load_and_label("train.csv")
valid = load_and_label("valid.csv")
test = load_and_label("test.csv")

balance = train["Label"].value_counts(normalize=True).rename({0: "fake", 1: "real"})
assert 0.35 < balance["fake"] < 0.5, "Fake-class share outside the expected ~44% range -- check label mapping"

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return " ".join(words)


train["clean_text"] = train["Statement"].apply(preprocess)
valid["clean_text"] = valid["Statement"].apply(preprocess)
test["clean_text"] = test["Statement"].apply(preprocess)

y_train, y_valid, y_test = train["Label"], valid["Label"], test["Label"]

TF-IDF text features (fit on train only, same params as the text-only pipeline)

In [3]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.9)

X_train_tfidf = tfidf.fit_transform(train["clean_text"])
X_valid_tfidf = tfidf.transform(valid["clean_text"])
X_test_tfidf = tfidf.transform(test["clean_text"])

print("TF-IDF shape:", X_train_tfidf.shape)

TF-IDF shape: (10240, 10000)


Metadata features -- no speaker, and with hashed speaker (fit on train only)

In [4]:
X_train_meta, X_valid_meta, X_test_meta, _ = build_metadata_features(
    train, valid, test, include_speaker=False
)
X_train_meta_spk, X_valid_meta_spk, X_test_meta_spk, _ = build_metadata_features(
    train, valid, test, include_speaker=True
)

print("Metadata (no speaker) shape:", X_train_meta.shape)
print("Metadata (+ hashed speaker) shape:", X_train_meta_spk.shape)

feature_sets = {
    "Text + metadata": (
        combine_text_and_metadata(X_train_tfidf, X_train_meta),
        combine_text_and_metadata(X_valid_tfidf, X_valid_meta),
        combine_text_and_metadata(X_test_tfidf, X_test_meta),
    ),
    "Text + metadata + speaker (hashed)": (
        combine_text_and_metadata(X_train_tfidf, X_train_meta_spk),
        combine_text_and_metadata(X_valid_tfidf, X_valid_meta_spk),
        combine_text_and_metadata(X_test_tfidf, X_test_meta_spk),
    ),
}
for name, (xtr, _, _) in feature_sets.items():
    print(name, "combined shape:", xtr.shape)

Metadata (no speaker) shape: (10240, 669)
Metadata (+ hashed speaker) shape: (10240, 733)
Text + metadata combined shape: (10240, 10669)
Text + metadata + speaker (hashed) combined shape: (10240, 10733)


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


GridSearch + evaluation -- same objective (macro-F1) and models as the text-only proposed pipeline

In [5]:
def train_and_evaluate(model, param_grid, X_train, y_train, X_valid, y_valid, X_test, y_test):
    grid = GridSearchCV(model, param_grid, cv=5, scoring="f1_macro", n_jobs=-1)
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    valid_metrics = evaluate_full(y_valid, best_model.predict(X_valid))
    test_metrics = evaluate_full(y_test, best_model.predict(X_test))
    return best_model, grid.best_params_, valid_metrics, test_metrics


def make_models():
    return [
        (
            "Logistic Regression",
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            {"C": [0.1, 1, 10], "solver": ["liblinear"], "class_weight": [None, "balanced"]},
        ),
        (
            "SVM",
            LinearSVC(random_state=RANDOM_STATE),
            {"C": [0.1, 1, 10], "class_weight": [None, "balanced"]},
        ),
        ("Naive Bayes", MultinomialNB(), {"alpha": [0.1, 0.5, 1.0]}),
        (
            "Random Forest",
            RandomForestClassifier(random_state=RANDOM_STATE),
            {"n_estimators": [100, 200], "max_depth": [None, 10], "min_samples_split": [2, 5]},
        ),
        (
            "XGBoost",
            XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
            {"n_estimators": [100, 200], "max_depth": [3, 6], "learning_rate": [0.01, 0.1]},
        ),
    ]

In [6]:
rows = []
for feature_set_name, (xtr, xva, xte) in feature_sets.items():
    for name, model, params in make_models():
        print(f"\nTraining {name} on [{feature_set_name}]...")
        best_model, best_params, valid_m, test_m = train_and_evaluate(
            model, params, xtr, y_train, xva, y_valid, xte, y_test
        )
        print("Best params:", best_params)
        print_report(f"{name} [{feature_set_name}]", y_test, best_model.predict(xte))
        rows.append(
            {
                "Pipeline": "Proposed",
                "Method": feature_set_name,
                "Model": name,
                "Valid Accuracy": valid_m["accuracy"],
                "Valid Macro-F1": valid_m["macro_f1"],
                "Valid Fake F1": valid_m["fake_f1"],
                "Test Accuracy": test_m["accuracy"],
                "Test Macro-F1": test_m["macro_f1"],
                "Test Fake Precision": test_m["fake_precision"],
                "Test Fake Recall": test_m["fake_recall"],
                "Test Fake F1": test_m["fake_f1"],
                "Test Real F1": test_m["real_f1"],
                "Test Confusion Matrix": test_m["confusion_matrix"],
                "Best Params": best_params,
            }
        )

metadata_results = pd.DataFrame(rows)


Training Logistic Regression on [Text + metadata]...


Best params: {'C': 1, 'class_weight': 'balanced', 'solver': 'liblinear'}

Logistic Regression [Text + metadata]
[[314 239]
 [229 485]]
              precision    recall  f1-score   support

        fake      0.578     0.568     0.573       553
        real      0.670     0.679     0.675       714

    accuracy                          0.631      1267
   macro avg      0.624     0.624     0.624      1267
weighted avg      0.630     0.631     0.630      1267


Training SVM on [Text + metadata]...


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/rav

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best params: {'C': 0.1, 'class_weight': 'balanced'}

SVM [Text + metadata]
[[323 230]
 [232 482]]
              precision    recall  f1-score   support

        fake      0.582     0.584     0.583       553
        real      0.677     0.675     0.676       714

    accuracy                          0.635      1267
   macro avg      0.629     0.630     0.630      1267
weighted avg      0.636     0.635     0.635      1267


Training Naive Bayes on [Text + metadata]...
Best params: {'alpha': 0.5}

Naive Bayes [Text + metadata]
[[305 248]
 [183 531]]
              precision    recall  f1-score   support

        fake      0.625     0.552     0.586       553
        real      0.682     0.744     0.711       714

    accuracy                          0.660      1267
   macro avg      0.653     0.648     0.649      1267
weighted avg      0.657     0.660     0.657      1267


Training Random Forest on [Text + metadata]...


...

Best params: {'C': 1, 'class_weight': 'balanced', 'solver': 'liblinear'}

Logistic Regression [Text + metadata + speaker (hashed)]
[[316 237]
 [235 479]]
              precision    recall  f1-score   support

        fake      0.574     0.571     0.572       553
        real      0.669     0.671     0.670       714

    accuracy                          0.627      1267
   macro avg      0.621     0.621     0.621      1267
weighted avg      0.627     0.627     0.627      1267


Training SVM on [Text + metadata + speaker (hashed)]...


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/rav

Best params: {'C': 0.1, 'class_weight': 'balanced'}

SVM [Text + metadata + speaker (hashed)]
[[314 239]
 [232 482]]
              precision    recall  f1-score   support

        fake      0.575     0.568     0.571       553
        real      0.669     0.675     0.672       714

    accuracy                          0.628      1267
   macro avg      0.622     0.621     0.622      1267
weighted avg      0.628     0.628     0.628      1267


Training Naive Bayes on [Text + metadata + speaker (hashed)]...
Best params: {'alpha': 0.5}

Naive Bayes [Text + metadata + speaker (hashed)]
[[303 250]
 [188 526]]
              precision    recall  f1-score   support

        fake      0.617     0.548     0.580       553
        real      0.678     0.737     0.706       714

    accuracy                          0.654      1267
   macro avg      0.647     0.642     0.643      1267
weighted avg      0.651     0.654     0.651      1267


Training Random Forest on [Text + metadata + speaker (hashed)]

Best params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}

Random Forest [Text + metadata + speaker (hashed)]
[[242 311]
 [135 579]]
              precision    recall  f1-score   support

        fake      0.642     0.438     0.520       553
        real      0.651     0.811     0.722       714

    accuracy                          0.648      1267
   macro avg      0.646     0.624     0.621      1267
weighted avg      0.647     0.648     0.634      1267


Training XGBoost on [Text + metadata + speaker (hashed)]...


Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}

XGBoost [Text + metadata + speaker (hashed)]
[[249 304]
 [156 558]]
              precision    recall  f1-score   support

        fake      0.615     0.450     0.520       553
        real      0.647     0.782     0.708       714

    accuracy                          0.637      1267
   macro avg      0.631     0.616     0.614      1267
weighted avg      0.633     0.637     0.626      1267



Merge with the text-only (Chi2/MI) and baseline/Dummy results from Phase 1 -- one comparable
table, same metric set throughout (I-4). This is also the raw material for the Phase 2
ablation (`text -> +metadata -> +feature selection -> +tuning`).

In [7]:
phase1_results = pd.read_csv("model_comparison_results_v2.csv")

all_results = pd.concat([phase1_results, metadata_results], ignore_index=True)
all_results = all_results.sort_values("Test Macro-F1", ascending=False)
all_results.to_csv("model_comparison_results_v3_metadata.csv", index=False)
all_results[["Pipeline", "Method", "Model", "Test Accuracy", "Test Macro-F1", "Test Fake F1"]]

,Pipeline,Method,Model,Test Accuracy,Test Macro-F1,Test Fake F1
20,Proposed,Text + metadata,Naive Bayes,0.659826,0.648647,0.585975
25,Proposed,Text + metadata + speaker (hashed),Naive Bayes,0.654301,0.643250,0.580460
19,Proposed,Text + metadata,SVM,0.635359,0.629525,0.583032
18,Proposed,Text + metadata,Logistic Regression,0.630624,0.623770,0.572993
22,Proposed,Text + metadata,XGBoost,0.643252,0.621913,0.532091
24,Proposed,Text + metadata + speaker (hashed),SVM,0.628256,0.621603,0.571429
23,Proposed,Text + metadata + speaker (hashed),Logistic Regression,0.627466,0.621197,0.572464
26,Proposed,Text + metadata + speaker (hashed),Random Forest,0.647987,0.621188,0.520430
0,Proposed,Mutual Information,Logistic Regression,0.621942,0.619175,0.586713
27,Proposed,Text + metadata + speaker (hashed),XGBoost,0.636938,0.613977,0.519833


Isolate the metadata effect: best text-only vs. best text+metadata vs. best text+metadata+speaker,
same model family where possible.

In [8]:
methods_of_interest = [
    "Chi-square",
    "Mutual Information",
    "Text + metadata",
    "Text + metadata + speaker (hashed)",
]
summary = (
    all_results[all_results["Method"].isin(methods_of_interest)]
    .sort_values("Test Macro-F1", ascending=False)
    .groupby("Method", sort=False)
    .first()[["Model", "Test Accuracy", "Test Macro-F1", "Test Fake F1"]]
)
summary.reindex(methods_of_interest)

,Model,Test Accuracy,Test Macro-F1,Test Fake F1
Method,,,,
Chi-square,Logistic Regression,0.594317,0.589934,0.547535
Mutual Information,Logistic Regression,0.621942,0.619175,0.586713
Text + metadata,Naive Bayes,0.659826,0.648647,0.585975
Text + metadata + speaker (hashed),Naive Bayes,0.654301,0.643250,0.580460
